In [1]:
# load the csv file
import pandas as pd

df = pd.read_csv('./scrap_data.csv', on_bad_lines='skip')
df = df.dropna(subset = ["link", "ASIN"])
products = df[['ASIN', 'link']].drop_duplicates().values.tolist()


In [2]:
import requests
import random
from bs4 import BeautifulSoup
from datetime import date
import time

In [ ]:
# MySQL Connection
import mysql.connector

# 1) Connect to MySQL server first (without database)
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Enter your password"
)
cursor = conn.cursor()

# 2) Create database if it does not exist, then select it
cursor.execute("CREATE DATABASE IF NOT EXISTS amazon_scraping")
cursor.execute("USE amazon_scraping")


In [4]:
# INSERT IGNORE skips silently if ASIN + scrape_date already exists
insert_query = """
    INSERT IGNORE INTO amazon_products
    (ASIN, title, price, rating, link, scrape_date)
    VALUES (%s, %s, %s, %s, %s, %s)
"""


#### Session + anti-bot setup

In [5]:
USER_AGENTS = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/121.0.0.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Chrome/120.0.0.0",
        "Mozilla/5.0 (X11; Linux x86_64) Firefox/118.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.6167.140 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.6099.216 Safari/537.36",
        "Mozilla/5.0 (X11; Linux x86_64; rv:118.0) Gecko/20100101 Firefox/118.0"]

def get_headers():
    return {
        'User-Agent' : random.choice(USER_AGENTS),
        'Accept-Language' : 'en-US,en;q=0.9',
        'Accept' : 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        "Accept-Encoding" : "gzip, deflate, br",
        "Connection" : "keep-alive",
        "DNT" : "1",
        "Upgrade-Insecure-Requests" : "1"
    }
    
def is_blocked(html):
    return any(x in html.lower() for x in ["captcha", "robot check", "enter the characters"])

today = date.today()

session = requests.Session()
session.headers.update(get_headers())

# Warm-up
session.get("https://www.amazon.in")
time.sleep(random.uniform(4,7))

session.post(
    "https://www.amazon.in/gp/delivery/ajax/address-change.html",
    data={
        "locationType": "LOCATION_INPUT",
        "zipCode": "700121",   # any valid Indian pincode
        "deviceType": "web"
    },
    headers={"X-Requested-With": "XMLHttpRequest"}
)


<Response [200]>

### Scrape product detail page + append

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import mysql.connector.pooling

# ── Table setup (runs once, safe to re-run) ─────────────────────────────────
cursor.execute("""
    CREATE TABLE IF NOT EXISTS amazon_products (
        id          INT AUTO_INCREMENT PRIMARY KEY,
        ASIN        VARCHAR(20)  NOT NULL,
        title       TEXT,
        price       VARCHAR(50),
        rating      VARCHAR(20),
        link        TEXT,
        scrape_date DATE,
        created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE KEY  uniq_asin_date (ASIN, scrape_date)
    )
""")
conn.commit()

# ── Thread-safe MySQL connection pool (one conn per worker) ──────────────────
db_pool = mysql.connector.pooling.MySQLConnectionPool(
    pool_name = "scraper_pool",
    pool_size  = 4,
    host       = "localhost",
    user       = "root",
    password   = "Wasim2001@",
    database   = "amazon_scraping"
)

# ── Per-thread scrape function ────────────────────────────────────────────────
def make_session():
    """Create a fresh session with randomised headers."""
    s = requests.Session()
    s.headers.update(get_headers())
    # Lightweight warm-up to build a cookie jar
    try:
        s.get("https://www.amazon.in", timeout=10)
        time.sleep(random.uniform(1, 2))
    except Exception:
        pass
    return s

def scrape_product(args):
    idx, asin, link = args
    session = make_session()
    backoff = 30                         # seconds; doubles on each block

    for attempt in range(3):
        # Fresh headers + referer on every attempt
        session.headers.update({
            **get_headers(),
            "Referer": "https://www.amazon.in/s?k=" + asin
        })

        time.sleep(random.uniform(4, 9))  # base delay per request

        try:
            response = session.get(link, timeout=25)
        except Exception as e:
            return idx, asin, f"request_error ({e})"

        if response.status_code == 200 and not is_blocked(response.text):
            break  # success

        print(f"  ⚠ [{asin}] blocked (attempt {attempt + 1}/3) → waiting {backoff}s")
        time.sleep(backoff)
        backoff *= 2            # 30 → 60 → 120
        session = make_session() # new session after each block
    else:
        return idx, asin, "blocked after 3 attempts"

    # ── Parse ────────────────────────────────────────────────────────────────
    soup      = BeautifulSoup(response.content, "html.parser")
    title_el  = soup.select_one("#productTitle")
    price_el  = soup.select_one("span.a-price-whole")
    rating_el = soup.select_one("span.a-icon-alt")

    title  = title_el.text.strip()        if title_el  else None
    price  = price_el.text.strip()        if price_el  else None
    rating = rating_el.text.split()[0]    if rating_el else None

    # ── Insert (thread-safe via pool) ────────────────────────────────────────
    db_conn = db_pool.get_connection()
    db_cur  = db_conn.cursor()
    db_cur.execute(insert_query, (asin, title, price, rating, link, today))
    db_conn.commit()
    db_cur.close()
    db_conn.close()

    return idx, asin, "saved"

# ── Run 3 parallel workers ────────────────────────────────────────────────────
tasks = [(i + 1, asin, link) for i, (asin, link) in enumerate(products)]

print(f"Starting scrape: {len(tasks)} products  |  3 parallel workers\n")

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(scrape_product, t): t for t in tasks}
    for future in as_completed(futures):
        idx, asin, status = future.result()
        icon = "✔" if status == "saved" else "⚠"
        print(f"{idx:>4}) {icon}  {asin}  →  {status}")

print("\nDone.")


Starting scrape: 465 products  |  3 parallel workers

   3) ✔  B0FQFNQ5LX  →  saved
   1) ✔  B0DXQH1DBS  →  saved
   2) ✔  B0FQFQF6D1  →  saved
   5) ✔  B0FQF2ZJWT  →  saved
   4) ✔  B0FQG8WM6R  →  saved
   6) ✔  B0FNMN6D5K  →  saved
